In [ ]:
%load_ext autoreload
%autoreload 2
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchdyn.core import NeuralODE
from torchvision import datasets, transforms
from torchvision.transforms import ToPILImage
from torchvision.utils import make_grid
from tqdm import tqdm

from torchcfm.conditional_flow_matching import (
    AnisoParamsND,
    AnisotropicHarmonicNDConditionalFlowMatcher,
    ExactOptimalTransportAnisotropicHarmonicNDConditionalFlowMatcher,
    ExactOptimalTransportHarmonicConditionalFlowMatcher,
)
from torchcfm.models.unet import UNetModel

savedir = "models/mnist_aniso"
os.makedirs(savedir, exist_ok=True)

## Load MNIST and fit PCA parameters

`AnisoParamsND.from_data` runs a full SVD on the training set to obtain the
principal-component rotation matrix and assigns per-eigendirection frequencies
linearly from `omega_base` (highest-variance PC) to `omega_base * omega_ratio`
(lowest-variance PC).  All frequencies must satisfy sin(ω) > 0.

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 128
n_epochs = 3

trainset = datasets.MNIST(
    "../data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
    ),
)
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, drop_last=True
)

# Fit AnisoParamsND from the normalized training images
print("Computing PCA from MNIST training images (may take ~30 s)...")
all_images = trainset.data.float().view(-1, 784) / 255.0 * 2 - 1  # → [-1, 1]
aniso_params = AnisoParamsND.from_data(
    all_images.numpy(), omega_base=0.8, omega_ratio=2.0
)
print(
    f"AnisoParamsND fitted: d={len(aniso_params.omegas)}, "
    f"ω ∈ [{aniso_params.omegas.min():.3f}, {aniso_params.omegas.max():.3f}]"
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 23.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.99MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.91MB/s]


Computing PCA from MNIST training images (may take ~30 s)...


---
### Baseline: Harmonic OT-CFM

Isotropic harmonic paths with a single scalar frequency ω = π/2, coupled via
exact optimal transport (Euclidean cost).

In [ ]:
sigma = 0.0
omega = math.pi / 2

model_h = UNetModel(dim=(1, 28, 28), num_channels=32, num_res_blocks=1).to(device)
optimizer_h = torch.optim.Adam(model_h.parameters())
FM_h = ExactOptimalTransportHarmonicConditionalFlowMatcher(sigma=sigma, omega=omega)

for epoch in range(n_epochs):
    for i, data in tqdm(enumerate(train_loader), desc=f"Epoch {epoch + 1}/{n_epochs}"):
        optimizer_h.zero_grad()
        x1 = data[0].to(device)
        x0 = torch.randn_like(x1)
        t, xt, ut = FM_h.sample_location_and_conditional_flow(x0, x1)
        vt = model_h(t, xt)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer_h.step()

torch.save(model_h.state_dict(), f"{savedir}/harmonic_otcfm.pt")
print("Saved harmonic OT-CFM model.")

In [ ]:
node_h = NeuralODE(model_h, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)
with torch.no_grad():
    traj_h = node_h.trajectory(
        torch.randn(100, 1, 28, 28, device=device),
        t_span=torch.linspace(0, 1, 2, device=device),
    )
grid_h = make_grid(
    traj_h[-1, :100].clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
plt.figure(figsize=(10, 10))
plt.title("Harmonic OT-CFM (baseline)")
plt.imshow(ToPILImage()(grid_h))
plt.axis("off")
plt.tight_layout()

---
### Anisotropic Harmonic CFM

Per-eigencomponent sinusoidal interpolation in the PCA eigenbasis of the MNIST
training distribution.  High-variance directions (stroke structure) get low ω
(gentle, near-linear paths); low-variance directions (fine detail) get high ω
(tighter sinusoidal paths that decay quickly).

$$\psi_t = R^\top \left(\frac{\sin(\omega(1-t))}{\sin\omega}\,\tilde{x}_0
         + \frac{\sin(\omega t)}{\sin\omega}\,\tilde{x}_1\right) + \mu$$

where $\tilde{x} = R(x_{\text{flat}} - \mu)$ are PCA coordinates.

In [ ]:
model_a = UNetModel(dim=(1, 28, 28), num_channels=32, num_res_blocks=1).to(device)
optimizer_a = torch.optim.Adam(model_a.parameters())
FM_a = AnisotropicHarmonicNDConditionalFlowMatcher(sigma=0.0, aniso_params=aniso_params)

for epoch in range(n_epochs):
    for i, data in tqdm(enumerate(train_loader), desc=f"Epoch {epoch + 1}/{n_epochs}"):
        optimizer_a.zero_grad()
        x1 = data[0].to(device)
        x0 = torch.randn_like(x1)
        t, xt, ut = FM_a.sample_location_and_conditional_flow(x0, x1)
        vt = model_a(t, xt)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer_a.step()

torch.save(model_a.state_dict(), f"{savedir}/aniso_harmonic_cfm.pt")
print("Saved anisotropic harmonic CFM model.")

In [ ]:
node_a = NeuralODE(model_a, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)
with torch.no_grad():
    traj_a = node_a.trajectory(
        torch.randn(100, 1, 28, 28, device=device),
        t_span=torch.linspace(0, 1, 2, device=device),
    )
grid_a = make_grid(
    traj_a[-1, :100].clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
plt.figure(figsize=(10, 10))
plt.title("Anisotropic Harmonic CFM")
plt.imshow(ToPILImage()(grid_a))
plt.axis("off")
plt.tight_layout()

---
### Action-OT Anisotropic Harmonic CFM

The strongest variant: anisotropic harmonic paths **plus** optimal transport coupling
where the transport cost is the anisotropic action S(x₀, x₁) in the PCA eigenbasis:

$$S(x_0, x_1) = \sum_k \frac{\omega_k}{2\sin\omega_k}
  \left[(\tilde{x}_0^k)^2 + (\tilde{x}_1^k)^2\right]\cos\omega_k - 2\tilde{x}_0^k \tilde{x}_1^k$$

where $\tilde{x}^k = [R(x_{\text{flat}} - \mu)]_k$ are PCA coordinates.

High-variance PCA directions (low ω, stroke structure) incur **low** transport cost →
the coupling is flexible along those directions.  Low-variance directions (high ω,
fine detail) **penalise** cross-gap transport → the coupling is tight there.

This means the OT plan is geometry-aware: it matches source and target images that
are close in the anisotropic eigenbasis, not just in raw pixel space.

In [ ]:
model_aot = UNetModel(dim=(1, 28, 28), num_channels=32, num_res_blocks=1).to(device)
optimizer_aot = torch.optim.Adam(model_aot.parameters())
FM_aot = ExactOptimalTransportAnisotropicHarmonicNDConditionalFlowMatcher(
    sigma=0.0, aniso_params=aniso_params
)

for epoch in range(n_epochs):
    for i, data in tqdm(enumerate(train_loader), desc=f"Epoch {epoch + 1}/{n_epochs}"):
        optimizer_aot.zero_grad()
        x1 = data[0].to(device)
        x0 = torch.randn_like(x1)
        t, xt, ut = FM_aot.sample_location_and_conditional_flow(x0, x1)
        vt = model_aot(t, xt)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer_aot.step()

torch.save(model_aot.state_dict(), f"{savedir}/aniso_ot_harmonic_cfm.pt")
print("Saved action-OT anisotropic harmonic CFM model.")

In [ ]:
node_aot = NeuralODE(model_aot, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)
with torch.no_grad():
    traj_aot = node_aot.trajectory(
        torch.randn(100, 1, 28, 28, device=device),
        t_span=torch.linspace(0, 1, 2, device=device),
    )
grid_aot = make_grid(
    traj_aot[-1, :100].clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
plt.figure(figsize=(10, 10))
plt.title("Action-OT Anisotropic Harmonic CFM")
plt.imshow(ToPILImage()(grid_aot))
plt.axis("off")
plt.tight_layout()

---
### Comparison: All Three Methods

Side-by-side comparison of generated samples after 3 epochs of training.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(30, 10))
for ax, grid, title in zip(
    axes,
    [grid_h, grid_a, grid_aot],
    [
        "Harmonic OT-CFM\n(isotropic ω=π/2, Euclidean OT)",
        "Anisotropic Harmonic CFM\n(per-PC ω, random coupling)",
        "Action-OT Anisotropic Harmonic CFM\n(per-PC ω, action-OT coupling)",
    ],
):
    ax.imshow(ToPILImage()(grid))
    ax.set_title(title, fontsize=14)
    ax.axis("off")
plt.tight_layout()

---
### PCA Frequency Spectrum

Visualise how frequency ω is assigned to each PCA component.  The first
component (highest variance, ~stroke direction) gets the lowest ω; the last
component (lowest variance, fine pixel noise) gets the highest ω.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(aniso_params.omegas)
axes[0].set_xlabel("PCA component index")
axes[0].set_ylabel("ω (frequency)")
axes[0].set_title("Frequency schedule across PCA components\n(low index = high variance = low ω)")

# Variance captured by each PC (proxy: singular values are stored implicitly in Vt ordering)
# Re-project training data to get explained variance
R = torch.tensor(aniso_params.eigvecs, dtype=torch.float32)
center = torch.tensor(aniso_params.center, dtype=torch.float32)
subset = all_images[:2000]  # use a subset for speed
projected = (subset - center) @ R.T  # [N, d]
variances = projected.var(0).numpy()

axes[1].semilogy(variances)
axes[1].set_xlabel("PCA component index")
axes[1].set_ylabel("Variance (log scale)")
axes[1].set_title("Variance explained by each PCA component")

plt.tight_layout()

---
### Top PCA Components

The leading PCA eigenvectors of MNIST capture global stroke structure.  These
are the directions assigned the lowest (gentlest) frequencies.

In [ ]:
n_show = 16
components = torch.tensor(aniso_params.eigvecs[:n_show], dtype=torch.float32)  # [16, 784]
components = components.view(n_show, 1, 28, 28)
# Normalise each component independently for display
components = (components - components.flatten(1).min(1).values[:, None, None, None])
components = components / components.flatten(1).max(1).values[:, None, None, None]

grid_pc = make_grid(components, nrow=8, padding=2)
plt.figure(figsize=(12, 4))
plt.title("Top 16 PCA eigenvectors of MNIST (reshaped to 28×28)")
plt.imshow(ToPILImage()(grid_pc))
plt.axis("off")
plt.tight_layout()